# Inventory & Supply Chain Planning Analytics

**Vivek Kumar** | Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter, SQL, and Power BI

Planning-oriented analysis of daily SKU and warehouse records. The supplied data has no stockouts flagged and no dependable product category or geographic dimension, so the analysis focuses on demand, forecast error, reorder exposure, and inventory coverage.

## 1. Load and validate the source

Install dependencies from `requirements.txt` first. Run this notebook from the repository directory so relative paths resolve.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.analysis import load_data, enrich_data, forecast_metrics, build_outputs

DATA_PATH = Path('Inventory management dataset.csv')
df = enrich_data(load_data(DATA_PATH))
df.head()

In [ ]:
profile = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'distinct_values': df.nunique(),
})
print(f'Rows: {len(df):,} | columns: {df.shape[1]} | duplicate rows: {df.duplicated().sum():,}')
print(f'Date range: {df.Date.min():%Y-%m-%d} to {df.Date.max():%Y-%m-%d}')
print(f'SKUs: {df.SKU_ID.nunique()} | warehouses: {df.Warehouse_ID.nunique()} | suppliers: {df.Supplier_ID.nunique()}')
profile

## 2. Demand and forecast quality

Forecast error is actual sales minus forecast. Positive bias means under-forecasting. WAPE aggregates absolute errors over total actual sales and avoids the undefined per-row percentage when actual sales are zero. It is retrospective accuracy on this extract, not a future performance guarantee.

In [ ]:
metrics = forecast_metrics(df)
pd.Series(metrics, name='Value').to_frame()

In [ ]:
monthly = df.assign(Month=df.Date.dt.to_period('M').astype(str)).groupby('Month', as_index=False).agg(
    Units_Sold=('Units_Sold', 'sum'), Demand_Forecast=('Demand_Forecast', 'sum'))
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly.Month, monthly.Units_Sold, marker='o', label='Actual units')
ax.plot(monthly.Month, monthly.Demand_Forecast, marker='o', label='Forecast units')
ax.set(title='Monthly Demand and Forecast | 2024', xlabel='Month', ylabel='Units')
ax.tick_params(axis='x', rotation=45); ax.legend(); ax.grid(axis='y', alpha=.25); fig.tight_layout()

## 3. Inventory planning review

An observation at or below its supplied reorder point is a screening signal. Estimated days of cover uses final-day inventory divided by the item's average daily sales across the extract. It is not a policy recommendation: it ignores forecast shape, variability, safety stock, open purchase orders, and service targets.

In [ ]:
planning = build_outputs(df, Path('outputs'))['sku_warehouse_planning']
attention = planning.Planning_Attention.value_counts()
print(f"Rows at/below reorder point: {df.At_or_Below_Reorder_Point.sum():,} ({df.At_or_Below_Reorder_Point.mean():.1%})")
print(f"Stockout flag rate: {df.Stockout_Flag.mean():.1%}; observed flag values: {sorted(df.Stockout_Flag.unique())}")
display(attention.to_frame('SKU-warehouse pairs'))
planning.head(15)

In [ ]:
plot_data = planning.sort_values('Estimated_Days_of_Cover').head(20)
plt.figure(figsize=(10, 7))
sns.barplot(data=plot_data, y='SKU_ID', x='Estimated_Days_of_Cover', hue='Warehouse_ID', dodge=False)
plt.title('Lowest Estimated Days of Inventory Cover by SKU and Warehouse')
plt.xlabel('Days (final inventory / historical average daily sales)'); plt.ylabel('SKU')
plt.tight_layout()

## 4. Commercial view and interpretation

Revenue and gross profit are calculated from recorded unit values. Gross profit is before freight, labor, discounts outside the unit price, and overhead. Region labels vary within a SKU/warehouse over time, so regional comparisons are descriptive only. Promotion comparisons are observational and should not be interpreted as causal lift.

In [ ]:
commercial = df.groupby('Promotion_Flag', as_index=False).agg(
    Units_Sold=('Units_Sold', 'mean'), Revenue=('Revenue', 'mean'),
    Gross_Profit=('Gross_Profit', 'mean'), Records=('Date', 'count'))
commercial['Promotion'] = commercial.Promotion_Flag.map({0: 'No promotion', 1: 'Promotion'})
commercial

## 5. Reproducible outputs and handoff

The analysis script writes monthly demand, region summaries, and SKU/warehouse review tables to `outputs/`. See `docs/data_dictionary.md` for the full field descriptions, formulas, caveats, and source audit; see `docs/power_bi_rebuild.md` for the proposed dashboard model. The included PBIX and screenshot are upstream artifacts and do not represent this dataset without a rebuild.